# 02 Model Training (V2)
Train and compare four models: TF-IDF+LR, TF-IDF+LinearSVC (word+char FeatureUnion), ToxiGen-RoBERTa (frozen)+LR, and fine-tuned MiniLM. All run on Colab GPU.

In [26]:
# --- Colab setup ---
from google.colab import drive
drive.mount('/content/drive')
import os
os.chdir('/content/drive/My Drive/BT5151_toxic_comment_agent/experiments')

# --- Install dependencies ---
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    "transformers", "datasets", "evaluate", "accelerate",
    "sentence-transformers", "scikit-learn", "seaborn"])

# --- Imports ---
import json, pickle, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay,
    RocCurveDisplay, roc_curve,
)

import transformers
from transformers import (
    AutoTokenizer, AutoModel,
    AutoModelForSequenceClassification,
    TrainingArguments, Trainer,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
)
from datasets import Dataset
import evaluate as hf_evaluate

warnings.filterwarnings("ignore")

# --- Constants ---
RANDOM_STATE  = 42
PROCESSED_DIR = Path("processed_data")
MODELS_DIR    = Path("models")
FIGURES_DIR   = Path("figures")
MODELS_DIR.mkdir(exist_ok=True)
FIGURES_DIR.mkdir(exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

TFIDF_WORD_PARAMS = dict(analyzer="word", ngram_range=(1, 2), max_features=50_000)
TFIDF_CHAR_PARAMS = dict(analyzer="char_wb", ngram_range=(3, 5), max_features=30_000)
LR_C_GRID         = [0.1, 1.0, 5.0]
SVC_C_GRID        = [0.01, 0.1, 1.0]
THRESHOLD         = 0.5

TOXIGEN_MODEL_NAME = "tomh/toxigen_roberta"
MINILM_MODEL_NAME  = "sentence-transformers/all-MiniLM-L6-v2"
MINILM_MAX_LEN     = 128
METRIC_COLUMNS     = ["accuracy", "precision", "recall", "f1", "roc_auc"]

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Device: cuda


## Processed-Data Load

In [27]:
EXPECTED_COLUMNS = ["id", "comment_text_clean", "toxic_label"]

splits = {}
for split_name in ("train", "val", "test"):
    path = PROCESSED_DIR / f"{split_name}_set.csv"
    df   = pd.read_csv(path)
    assert list(df.columns[:3]) == EXPECTED_COLUMNS, f"{split_name}: unexpected columns {df.columns.tolist()}"
    assert df["comment_text_clean"].notna().all(), f"{split_name}: nulls in comment_text_clean"
    assert df["toxic_label"].isin([0, 1]).all(), f"{split_name}: non-binary toxic_label"
    splits[split_name] = df
    print(f"{split_name}: {len(df):,} rows | toxic rate: {df['toxic_label'].mean():.3f}")

train_set, val_set, test_set = splits["train"], splits["val"], splits["test"]

y_train = train_set["toxic_label"].values
y_val   = val_set["toxic_label"].values
y_test  = test_set["toxic_label"].values

train_texts = train_set["comment_text_clean"].tolist()
val_texts   = val_set["comment_text_clean"].tolist()
test_texts  = test_set["comment_text_clean"].tolist()

train: 127,656 rows | toxic rate: 0.102
val: 31,915 rows | toxic rate: 0.102
test: 63,978 rows | toxic rate: 0.098


## Shared Metric Helpers

In [28]:
def safe_roc_auc(y_true, y_score):
    y_true  = np.asarray(y_true).astype(int)
    y_score = np.asarray(y_score, dtype=float)
    if np.unique(y_true).size < 2:
        return float("nan")
    try:
        return float(roc_auc_score(y_true, y_score))
    except ValueError:
        return float("nan")

def compute_metrics(y_true, y_pred, y_score) -> dict:
    """Return dict with keys: accuracy, precision, recall, f1, roc_auc."""
    return {
        "accuracy":  float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall":    float(recall_score(y_true, y_pred, zero_division=0)),
        "f1":        float(f1_score(y_true, y_pred, zero_division=0)),
        "roc_auc":   safe_roc_auc(y_true, y_score),
    }

print("Metric helpers ready.")

Metric helpers ready.


## TF-IDF Feature Build (word + char FeatureUnion)
Combines word n-grams (1,2) and char_wb n-grams (3,5) to capture both normal vocabulary and deliberate misspellings (a$$, f*ck, st0p).

In [18]:
tfidf_union = FeatureUnion([
    ("word", TfidfVectorizer(**TFIDF_WORD_PARAMS)),
    ("char", TfidfVectorizer(**TFIDF_CHAR_PARAMS)),
])

X_train_tfidf = tfidf_union.fit_transform(train_texts)
X_val_tfidf   = tfidf_union.transform(val_texts)
X_test_tfidf  = tfidf_union.transform(test_texts)

print(f"TF-IDF feature matrix shape: train={X_train_tfidf.shape}, val={X_val_tfidf.shape}, test={X_test_tfidf.shape}")
# Expected: train=(127656, ~80000), val=(31915, ~80000), test=(63978, ~80000)

TF-IDF feature matrix shape: train=(127656, 80000), val=(31915, 80000), test=(63978, 80000)


## LR Tuning

In [19]:
lr_tuning_records = []
for c_val in LR_C_GRID:
    lr_cand = LogisticRegression(
        C=c_val, class_weight="balanced",
        max_iter=1000, random_state=RANDOM_STATE,
    )
    lr_cand.fit(X_train_tfidf, y_train)
    y_pred  = lr_cand.predict(X_val_tfidf)
    y_score = lr_cand.predict_proba(X_val_tfidf)[:, 1]
    metrics = compute_metrics(y_val, y_pred, y_score)
    lr_tuning_records.append({"model": "TF-IDF+LR", "C": c_val, **metrics})
    print(f"LR C={c_val}: F1={metrics['f1']:.4f} | AUC={metrics['roc_auc']:.4f}")

lr_tuning_df = pd.DataFrame(lr_tuning_records)
best_lr_row  = lr_tuning_df.loc[lr_tuning_df["f1"].idxmax()]
best_lr_c    = best_lr_row["C"]
print(f"\nBest LR C={best_lr_c} | Val F1={best_lr_row['f1']:.4f}")

LR C=0.1: F1=0.7264 | AUC=0.9727
LR C=1.0: F1=0.7743 | AUC=0.9791
LR C=5.0: F1=0.7911 | AUC=0.9776

Best LR C=5.0 | Val F1=0.7911


## LinearSVC Tuning
Linear SVM on the same TF-IDF FeatureUnion. Wrapped in CalibratedClassifierCV to produce probabilities for ROC-AUC.

In [20]:
svc_tuning_records = []
for c_val in SVC_C_GRID:
    base_svc  = LinearSVC(C=c_val, class_weight="balanced", max_iter=2000, random_state=RANDOM_STATE)
    svc_cand  = CalibratedClassifierCV(base_svc, cv=3)
    svc_cand.fit(X_train_tfidf, y_train)
    y_pred    = svc_cand.predict(X_val_tfidf)
    y_score   = svc_cand.predict_proba(X_val_tfidf)[:, 1]
    metrics   = compute_metrics(y_val, y_pred, y_score)
    svc_tuning_records.append({"model": "TF-IDF+LinearSVC", "C": c_val, **metrics})
    print(f"LinearSVC C={c_val}: F1={metrics['f1']:.4f} | AUC={metrics['roc_auc']:.4f}")

svc_tuning_df = pd.DataFrame(svc_tuning_records)
best_svc_row  = svc_tuning_df.loc[svc_tuning_df["f1"].idxmax()]
best_svc_c    = best_svc_row["C"]
print(f"\nBest LinearSVC C={best_svc_c} | Val F1={best_svc_row['f1']:.4f}")

LinearSVC C=0.01: F1=0.7520 | AUC=0.9714
LinearSVC C=0.1: F1=0.7953 | AUC=0.9797
LinearSVC C=1.0: F1=0.7965 | AUC=0.9770

Best LinearSVC C=1.0 | Val F1=0.7965


## ToxiGen-RoBERTa Embedding Generation (Frozen)
Extract mean-pooled last hidden state from `tomh/toxigen_roberta`. Embeddings cached as numpy arrays to avoid re-encoding during LR tuning.

In [21]:
def mean_pool(model_output, attention_mask):
    token_embeddings = model_output.last_hidden_state  # (B, T, H)
    mask = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    return (token_embeddings * mask).sum(1) / mask.sum(1)

def encode_texts(texts, tokenizer, model, batch_size=64, max_length=128):
    all_embeddings = []
    model.eval()
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch = texts[i : i + batch_size]
            enc   = tokenizer(batch, truncation=True, padding=True,
                              max_length=max_length, return_tensors="pt").to(DEVICE)
            out   = model(**enc)
            emb   = mean_pool(out, enc["attention_mask"]).cpu().numpy()
            all_embeddings.append(emb)
    return np.vstack(all_embeddings)

toxigen_tokenizer = AutoTokenizer.from_pretrained(TOXIGEN_MODEL_NAME)
toxigen_model     = AutoModel.from_pretrained(TOXIGEN_MODEL_NAME).to(DEVICE)

print("Encoding train set with ToxiGen-RoBERTa...")
toxigen_train_emb = encode_texts(train_texts, toxigen_tokenizer, toxigen_model)
print("Encoding val set...")
toxigen_val_emb   = encode_texts(val_texts,   toxigen_tokenizer, toxigen_model)
print("Encoding test set...")
toxigen_test_emb  = encode_texts(test_texts,  toxigen_tokenizer, toxigen_model)

print(f"ToxiGen embedding shapes: train={toxigen_train_emb.shape}, val={toxigen_val_emb.shape}")
# Expected: train=(127656, 768), val=(31915, 768), test=(63978, 768)

# Cache to disk
np.save(MODELS_DIR / "toxigen_train_emb.npy", toxigen_train_emb)
np.save(MODELS_DIR / "toxigen_val_emb.npy",   toxigen_val_emb)
np.save(MODELS_DIR / "toxigen_test_emb.npy",  toxigen_test_emb)
print("Embeddings cached.")

config.json:   0%|          | 0.00/790 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: tomh/toxigen_roberta
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
classifier.dense.bias           | UNEXPECTED | 
classifier.dense.weight         | UNEXPECTED | 
classifier.out_proj.bias        | UNEXPECTED | 
classifier.out_proj.weight      | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Encoding train set with ToxiGen-RoBERTa...
Encoding val set...
Encoding test set...
ToxiGen embedding shapes: train=(127656, 1024), val=(31915, 1024)
Embeddings cached.


## ToxiGen-RoBERTa + LR Tuning

In [22]:
toxigen_lr_tuning_records = []
for c_val in LR_C_GRID:
    tox_lr_cand = LogisticRegression(
        C=c_val, class_weight="balanced",
        max_iter=1000, random_state=RANDOM_STATE,
    )
    tox_lr_cand.fit(toxigen_train_emb, y_train)
    y_pred  = tox_lr_cand.predict(toxigen_val_emb)
    y_score = tox_lr_cand.predict_proba(toxigen_val_emb)[:, 1]
    metrics = compute_metrics(y_val, y_pred, y_score)
    toxigen_lr_tuning_records.append({"model": "ToxiGen-RoBERTa+LR", "C": c_val, **metrics})
    print(f"ToxiGen+LR C={c_val}: F1={metrics['f1']:.4f} | AUC={metrics['roc_auc']:.4f}")

toxigen_lr_df       = pd.DataFrame(toxigen_lr_tuning_records)
best_toxigen_lr_row = toxigen_lr_df.loc[toxigen_lr_df["f1"].idxmax()]
best_toxigen_lr_c   = best_toxigen_lr_row["C"]
print(f"\nBest ToxiGen+LR C={best_toxigen_lr_c} | Val F1={best_toxigen_lr_row['f1']:.4f}")

ToxiGen+LR C=0.1: F1=0.7253 | AUC=0.9771
ToxiGen+LR C=1.0: F1=0.7228 | AUC=0.9763
ToxiGen+LR C=5.0: F1=0.7225 | AUC=0.9760

Best ToxiGen+LR C=0.1 | Val F1=0.7253


## Fine-tune MiniLM (end-to-end classification)
Uses `AutoModelForSequenceClassification` with a 2-class head. Trained with HuggingFace Trainer for 3 epochs. Best checkpoint selected by validation F1.

In [23]:
minilm_tokenizer = AutoTokenizer.from_pretrained(MINILM_MODEL_NAME)

def tokenize_fn(examples):
    return minilm_tokenizer(
        examples["text"],
        truncation=True,
        padding=False,          # DataCollatorWithPadding handles this dynamically
        max_length=MINILM_MAX_LEN,
    )

def make_hf_dataset(texts, labels, desc):
    ds = Dataset.from_dict({"text": texts, "label": labels.tolist()})
    return ds.map(tokenize_fn, batched=True, remove_columns=["text"], desc=desc)

hf_train = make_hf_dataset(train_texts, y_train, "Tokenizing MiniLM train set")
hf_val   = make_hf_dataset(val_texts,   y_val,   "Tokenizing MiniLM val set")
print(f"HF train: {len(hf_train)} | HF val: {len(hf_val)}")

Tokenizing MiniLM train set:   0%|          | 0/127656 [00:00<?, ? examples/s]

Tokenizing MiniLM val set:   0%|          | 0/31915 [00:00<?, ? examples/s]

HF train: 127656 | HF val: 31915


In [24]:
f1_metric  = hf_evaluate.load("f1")
acc_metric = hf_evaluate.load("accuracy")

def compute_metrics_hf(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    f1  = f1_metric.compute(predictions=preds,  references=labels, average="binary")["f1"]
    acc = acc_metric.compute(predictions=preds, references=labels)["accuracy"]
    return {"f1": f1, "accuracy": acc}

In [32]:
minilm_model = AutoModelForSequenceClassification.from_pretrained(
    MINILM_MODEL_NAME, num_labels=2
).to(DEVICE)

training_args = TrainingArguments(
    output_dir=str(MODELS_DIR / "minilm_finetuned"),
    num_train_epochs=3,
    per_device_train_batch_size=128,
    per_device_eval_batch_size=128,
    warmup_ratio=0.1,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    fp16=(DEVICE == "cuda"),
    disable_tqdm=False,
    logging_steps=200,
    seed=RANDOM_STATE,
    report_to="none",
)

data_collator = DataCollatorWithPadding(tokenizer=minilm_tokenizer)

trainer = Trainer(
    model=minilm_model,
    args=training_args,
    train_dataset=hf_train,
    eval_dataset=hf_val,
    processing_class=minilm_tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics_hf,
)

trainer.train()
print("Fine-tuning complete.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     | 
------------------------+------------+-
embeddings.position_ids | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Epoch,Training Loss,Validation Loss,F1,Accuracy
1,0.091368,0.090299,0.830200,0.965753
2,0.073031,0.095230,0.824116,0.962431
3,0.052798,0.095448,0.833232,0.965659


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

Fine-tuning complete.


In [33]:
# Evaluate fine-tuned MiniLM on val set to record best val metrics
hf_val_pred  = trainer.predict(hf_val)
minilm_val_logits = hf_val_pred.predictions
minilm_val_preds  = np.argmax(minilm_val_logits, axis=-1)
minilm_val_scores = torch.softmax(torch.tensor(minilm_val_logits, dtype=torch.float32), dim=-1)[:, 1].numpy()
minilm_val_metrics = compute_metrics(y_val, minilm_val_preds, minilm_val_scores)
print(f"Fine-tuned MiniLM Val: F1={minilm_val_metrics['f1']:.4f} | AUC={minilm_val_metrics['roc_auc']:.4f}")

Fine-tuned MiniLM Val: F1=0.8332 | AUC=0.9859


## Validation Comparison

In [34]:
best_validation_records = [
    {"model_id": "lr",           "model": "TF-IDF+LR",             **{k: best_lr_row[k]         for k in METRIC_COLUMNS}, "C": best_lr_c},
    {"model_id": "linearsvc",    "model": "TF-IDF+LinearSVC",      **{k: best_svc_row[k]        for k in METRIC_COLUMNS}, "C": best_svc_c},
    {"model_id": "toxigen_lr",   "model": "ToxiGen-RoBERTa+LR",    **{k: best_toxigen_lr_row[k] for k in METRIC_COLUMNS}, "C": best_toxigen_lr_c},
    {"model_id": "minilm_ft",    "model": "MiniLM (fine-tuned)",   **minilm_val_metrics},
]

val_comparison_df = pd.DataFrame(best_validation_records)
print("\nValidation comparison:")
print(val_comparison_df[["model"] + METRIC_COLUMNS].to_string(index=False))


Validation comparison:
              model  accuracy  precision   recall       f1  roc_auc
          TF-IDF+LR  0.954316   0.739352 0.850539 0.791058 0.977555
   TF-IDF+LinearSVC  0.962557   0.890030 0.720801 0.796526 0.977031
 ToxiGen-RoBERTa+LR  0.929970   0.603272 0.909091 0.725261 0.977066
MiniLM (fine-tuned)  0.965659   0.822964 0.843760 0.833232 0.985880


## Refit (train+val)

In [35]:
train_val_set    = pd.concat([train_set, val_set], ignore_index=True)
train_val_texts  = train_val_set["comment_text_clean"].tolist()
y_train_val      = train_val_set["toxic_label"].values

# Refit TF-IDF union on full train+val
tfidf_union_final = FeatureUnion([
    ("word", TfidfVectorizer(**TFIDF_WORD_PARAMS)),
    ("char", TfidfVectorizer(**TFIDF_CHAR_PARAMS)),
])
X_train_val_tfidf  = tfidf_union_final.fit_transform(train_val_texts)
X_test_final_tfidf = tfidf_union_final.transform(test_texts)

# Refit LR
final_lr = LogisticRegression(C=best_lr_c, class_weight="balanced", max_iter=1000, random_state=RANDOM_STATE)
final_lr.fit(X_train_val_tfidf, y_train_val)

# Refit LinearSVC
final_svc = CalibratedClassifierCV(
    LinearSVC(C=best_svc_c, class_weight="balanced", max_iter=2000, random_state=RANDOM_STATE), cv=3
)
final_svc.fit(X_train_val_tfidf, y_train_val)

# Refit ToxiGen+LR on cached train+val embeddings
toxigen_train_val_emb = np.vstack([toxigen_train_emb, toxigen_val_emb])
final_toxigen_lr = LogisticRegression(C=best_toxigen_lr_c, class_weight="balanced", max_iter=1000, random_state=RANDOM_STATE)
final_toxigen_lr.fit(toxigen_train_val_emb, y_train_val)

# Fine-tuned MiniLM is already at its best checkpoint — no sklearn refit needed
print("Refit complete.")

Refit complete.


## Held-Out Test Evaluation

In [36]:
final_test_results = {}

# LR
lr_test_scores = final_lr.predict_proba(X_test_final_tfidf)[:, 1]
lr_test_preds  = (lr_test_scores >= THRESHOLD).astype(int)
final_test_results["lr"] = {
    "model": "TF-IDF+LR",
    **compute_metrics(y_test, lr_test_preds, lr_test_scores),
    "y_score": lr_test_scores, "y_pred": lr_test_preds,
}

# LinearSVC
svc_test_scores = final_svc.predict_proba(X_test_final_tfidf)[:, 1]
svc_test_preds  = (svc_test_scores >= THRESHOLD).astype(int)
final_test_results["linearsvc"] = {
    "model": "TF-IDF+LinearSVC",
    **compute_metrics(y_test, svc_test_preds, svc_test_scores),
    "y_score": svc_test_scores, "y_pred": svc_test_preds,
}

# ToxiGen+LR
tox_test_scores = final_toxigen_lr.predict_proba(toxigen_test_emb)[:, 1]
tox_test_preds  = (tox_test_scores >= THRESHOLD).astype(int)
final_test_results["toxigen_lr"] = {
    "model": "ToxiGen-RoBERTa+LR",
    **compute_metrics(y_test, tox_test_preds, tox_test_scores),
    "y_score": tox_test_scores, "y_pred": tox_test_preds,
}

# Fine-tuned MiniLM
hf_test     = make_hf_dataset(test_texts, y_test, "Tokenizing MiniLM test set")
test_pred   = trainer.predict(hf_test)
minilm_test_logits = test_pred.predictions
minilm_test_preds  = np.argmax(minilm_test_logits, axis=-1)
minilm_test_scores = torch.softmax(torch.tensor(minilm_test_logits, dtype=torch.float32), dim=-1)[:, 1].numpy()
final_test_results["minilm_ft"] = {
    "model": "MiniLM (fine-tuned)",
    **compute_metrics(y_test, minilm_test_preds, minilm_test_scores),
    "y_score": minilm_test_scores, "y_pred": minilm_test_preds,
}

# Print summary
test_summary = pd.DataFrame([
    {"model_id": k, **{m: v[m] for m in ["model"] + METRIC_COLUMNS}}
    for k, v in final_test_results.items()
])
print("\nHeld-out test results:")
print(test_summary[["model"] + METRIC_COLUMNS].to_string(index=False))

Tokenizing MiniLM test set:   0%|          | 0/63978 [00:00<?, ? examples/s]


Held-out test results:
              model  accuracy  precision   recall       f1  roc_auc
          TF-IDF+LR  0.883991   0.453564 0.922313 0.608090 0.964166
   TF-IDF+LinearSVC  0.927334   0.594588 0.802499 0.683073 0.963976
 ToxiGen-RoBERTa+LR  0.855091   0.398905 0.956912 0.563080 0.964425
MiniLM (fine-tuned)  0.907640   0.514844 0.927759 0.662208 0.972773


## Figure Export

In [37]:
# Confusion matrices
for model_id, result in final_test_results.items():
    fig, ax = plt.subplots(figsize=(5, 4))
    cm = confusion_matrix(y_test, result["y_pred"])
    ConfusionMatrixDisplay(cm, display_labels=["Clean", "Toxic"]).plot(ax=ax, colorbar=False)
    ax.set_title(result["model"])
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / f"{model_id}_confusion_matrix.png", dpi=150)
    plt.close(fig)
    print(f"Saved {model_id}_confusion_matrix.png")

# Combined ROC curve
fig, ax = plt.subplots(figsize=(7, 6))
for model_id, result in final_test_results.items():
    fpr, tpr, _ = roc_curve(y_test, result["y_score"])
    auc = result["roc_auc"]
    ax.plot(fpr, tpr, label=f"{result['model']} (AUC={auc:.3f})")
ax.plot([0, 1], [0, 1], "k--", linewidth=0.8)
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curves — All Models")
ax.legend(loc="lower right")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "combined_roc_curve.png", dpi=150)
plt.close(fig)
print("Saved combined_roc_curve.png")

# Summary metric comparison bar chart
fig, ax = plt.subplots(figsize=(10, 5))
plot_df = test_summary.set_index("model")[METRIC_COLUMNS]
plot_df.T.plot(kind="bar", ax=ax)
ax.set_ylim(0, 1)
ax.set_title("Test Metrics — All Models")
ax.set_ylabel("Score")
ax.legend(loc="lower right")
plt.xticks(rotation=0)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "summary_metric_comparison.png", dpi=150)
plt.close(fig)
print("Saved summary_metric_comparison.png")

Saved lr_confusion_matrix.png
Saved linearsvc_confusion_matrix.png
Saved toxigen_lr_confusion_matrix.png
Saved minilm_ft_confusion_matrix.png
Saved combined_roc_curve.png
Saved summary_metric_comparison.png


## Artifact Save

In [38]:
with open(MODELS_DIR / "tfidf_vectorizer.pkl",   "wb") as f: pickle.dump(tfidf_union_final, f)
with open(MODELS_DIR / "model_lr.pkl",            "wb") as f: pickle.dump(final_lr, f)
with open(MODELS_DIR / "model_linearsvc.pkl",     "wb") as f: pickle.dump(final_svc, f)
with open(MODELS_DIR / "model_toxigen_lr.pkl",    "wb") as f: pickle.dump(final_toxigen_lr, f)
print("Sklearn artifacts saved.")

Sklearn artifacts saved.


In [39]:
# trainer already saved checkpoints; explicitly save final best model
trainer.save_model(str(MODELS_DIR / "minilm_finetuned"))
minilm_tokenizer.save_pretrained(str(MODELS_DIR / "minilm_finetuned"))
print("Fine-tuned MiniLM saved to models/minilm_finetuned/")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Fine-tuned MiniLM saved to models/minilm_finetuned/


In [40]:
best_model_id  = test_summary.loc[test_summary["f1"].idxmax(), "model_id"]   # by test F1
best_model_row = test_summary[test_summary["model_id"] == best_model_id].iloc[0]

metadata = {
    "selected_model":    best_model_row["model"],
    "selected_model_id": best_model_id,
    "selection_reason":  (
        f"{best_model_row['model']} achieved the highest test F1={best_model_row['f1']:.4f}. "
        "Selected for content moderation: highest F1 balances precision and recall on the held-out set."
    ),
    "validation_metrics": {
        row["model_id"]: {m: row[m] for m in METRIC_COLUMNS}
        for _, row in val_comparison_df.iterrows()
        if "model_id" in row
    },
    "test_metrics": {
        model_id: {m: result[m] for m in METRIC_COLUMNS}
        for model_id, result in final_test_results.items()
    },
    "best_hyperparameters": {
        "lr":         {"C": best_lr_c},
        "linearsvc":  {"C": best_svc_c},
        "toxigen_lr": {"C": best_toxigen_lr_c},
        "minilm_ft":  {"epochs": 3, "batch_size": 32, "warmup_ratio": 0.1, "weight_decay": 0.01},
    },
    "artifact_paths": {
        "tfidf_vectorizer":  "models/tfidf_vectorizer.pkl",
        "model_lr":          "models/model_lr.pkl",
        "model_linearsvc":   "models/model_linearsvc.pkl",
        "model_toxigen_lr":  "models/model_toxigen_lr.pkl",
        "minilm_finetuned":  "models/minilm_finetuned/",
        "selected_model_metadata": "models/selected_model_metadata.json",
    },
    "tfidf_parameters": {
        "word": TFIDF_WORD_PARAMS,
        "char": TFIDF_CHAR_PARAMS,
    },
    "toxigen_model_name": TOXIGEN_MODEL_NAME,
    "minilm_model_name":  MINILM_MODEL_NAME,
    "random_state": RANDOM_STATE,
    "threshold":    THRESHOLD,
    "dataset_sizes": {
        "train": int(len(train_set)),
        "val":   int(len(val_set)),
        "train_plus_val": int(len(train_val_set)),
        "test":  int(len(test_set)),
    },
}

with open(MODELS_DIR / "selected_model_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print(f"\nSelected model: {metadata['selected_model']}")
print(f"Test F1: {best_model_row['f1']:.4f} | AUC: {best_model_row['roc_auc']:.4f}")
print("Metadata saved to models/selected_model_metadata.json")


Selected model: TF-IDF+LinearSVC
Test F1: 0.6831 | AUC: 0.9640
Metadata saved to models/selected_model_metadata.json


In [41]:
expected_artifacts = [
    MODELS_DIR / "tfidf_vectorizer.pkl",
    MODELS_DIR / "model_lr.pkl",
    MODELS_DIR / "model_linearsvc.pkl",
    MODELS_DIR / "model_toxigen_lr.pkl",
    MODELS_DIR / "minilm_finetuned",
    MODELS_DIR / "selected_model_metadata.json",
]
for path in expected_artifacts:
    assert Path(path).exists(), f"Missing: {path}"
    print(f"OK: {path}")

OK: models/tfidf_vectorizer.pkl
OK: models/model_lr.pkl
OK: models/model_linearsvc.pkl
OK: models/model_toxigen_lr.pkl
OK: models/minilm_finetuned
OK: models/selected_model_metadata.json
